In [ ]:
# Import the Roman HGA pointing helpers and create one reusable model instance
import RST_specific_functions as RST

model = RST.RomanHGAPointingModel()


In [ ]:
# Define yaw / pitch / roll attitude grids.
# Old two-value inputs still work, but tuples below use the full (yaw, pitch, roll) convention.
def make_attitude_grid(yaws=(0,), pitches=(36, 0, -36), rolls=(-15, 0, 15)):
    return [(yaw, pitch, roll) for yaw in yaws for pitch in pitches for roll in rolls]

all_for_attitudes = make_attitude_grid()
more_for_attitudes = make_attitude_grid(
    yaws=(0,),
    pitches=(36, 30, 24, 18, 12, 6, 0, -6, -12, -18, -24, -30, -36),
    rolls=(-15, -10, -5, 0, 5, 10, 15),
)

# Example yaw sweep you can turn on when you want yaw included in the trade space.
yaw_sweep_attitudes = make_attitude_grid(
    yaws=(-10, 0, 10),
    pitches=(36, 0, -36),
    rolls=(-15, 0, 15),
)


In [ ]:
# Keep the notebook interface light: define a target once, then solve attitudes with one helper.
def print_hga_inputs_across_for(target_vector, attitudes=all_for_attitudes, initial_guess=(0, 0)):
    print('HGA gimbal inputs across FOR for target vector:')
    print(target_vector)
    print()

    results = model.solve_across_attitudes(target_vector, attitudes, initial_guess=initial_guess)
    for result in results:
        attitude = result.attitude
        gimbal = result.gimbal_angles
        print(
            f'{attitude.label()}: '
            f'y_track={gimbal.y_track:.6f}, '
            f'x_track={gimbal.x_track:.6f}, '
            f'error={result.pointing_error:.6e}, '
            f'success={result.success}'
        )

    return results


def print_thermal_desktop_gimbal_exports(results, x_symbol_name, y_symbol_name):
    exports = model.format_thermal_desktop_gimbal_exports(results)

    print(x_symbol_name)
    print(exports.x_track)
    print('')
    print(y_symbol_name)
    print(exports.y_track)
    print('')

    return exports


In [ ]:
# Define one or more reference configurations.
# `target_attitude` can now include yaw directly as (yaw, pitch, roll).
reference_cases = [
    {
        'name': 'Wide configuration',
        'initial_gimbal': (10.013536790487828, -0.4368647026044624),
        'target_attitude': (0, 0, 0),
        'attitudes_to_solve': more_for_attitudes,
    },
    {
        'name': 'Skinny configuration',
        'initial_gimbal': (10, 15),
        'target_attitude': (0, 36, -15),
        'attitudes_to_solve': more_for_attitudes,
    },
    {
        'name': 'Shady configuration',
        'initial_gimbal': (-26, 0),
        'target_attitude': (0, 0, 0),
        'attitudes_to_solve': more_for_attitudes,
        'thermal_desktop_symbols': {
            'x_track': 'STOP_HG_Shady_Xtrack',
            'y_track': 'STOP_HG_Shady_Ytrack',
        },
    },
]

for case in reference_cases:
    print(case['name'])
    target_vector = model.define_target(case['initial_gimbal'], case['target_attitude'])
    results = print_hga_inputs_across_for(target_vector, case['attitudes_to_solve'])

    if 'thermal_desktop_symbols' in case:
        print_thermal_desktop_gimbal_exports(
            results,
            case['thermal_desktop_symbols']['x_track'],
            case['thermal_desktop_symbols']['y_track'],
        )

    print('')


In [ ]:
# Optional example: include yaw in the target definition and in the solved attitude sweep.
# Thermal Desktop export is intentionally not used here because the current export format assumes yaw = 0.
yaw_target_vector = model.define_target((10.013536790487828, -0.4368647026044624), (10, 12, -5), verbose=False)
print_hga_inputs_across_for(yaw_target_vector, yaw_sweep_attitudes)
